## Creating_features
Здесь мы займемся созданием признаков для моделей машинного обучения.

In [54]:
import pandas as pd

train_sparse = pd.read_csv('../../data/Cleared_data/train_sparse.csv', parse_dates=['date', 'created'])
train_sparse = train_sparse.sort_values(['station_id', 'date']).reset_index(drop=True)
train_sparse

,station_id,date,daily_sales_count,created,partner_name,employee_name,cities
0,277,2023-07-04,0.0,2023-07-04,Мегафон,Екатерина,Санкт-Петербург
1,277,2023-07-05,0.0,2023-07-04,Мегафон,Екатерина,Санкт-Петербург
2,277,2023-07-06,0.0,2023-07-04,Мегафон,Екатерина,Санкт-Петербург
3,277,2023-07-07,0.0,2023-07-04,Мегафон,Екатерина,Санкт-Петербург
4,277,2023-07-08,0.0,2023-07-04,Мегафон,Екатерина,Санкт-Петербург
...,...,...,...,...,...,...,...
1422218,3952,2026-07-29,0.0,2026-07-24,Beeline,Никита,Самара
1422219,3952,2026-07-30,0.0,2026-07-24,Beeline,Никита,Самара
1422220,3952,2026-07-31,0.0,2026-07-24,Beeline,Никита,Самара
1422221,3953,2026-07-30,1.0,2026-07-30,Inventive Group,Екатерина,Санкт-Петербург


In [55]:
train_sparse = train_sparse.sort_values(['station_id', 'date']).reset_index(drop=True)


train_sparse['target_next_30d'] = (
    train_sparse.groupby('station_id')['daily_sales_count']
    .transform(lambda x: x.rolling(30, min_periods=30).sum().shift(-30))
)

train_sparse.sort_values('date', ascending=False).tail(10)

,station_id,date,daily_sales_count,created,partner_name,employee_name,cities,target_next_30d
182084,843,2023-07-04,0.0,2023-07-04,МТС,Галина,Малый город,2.0
185137,870,2023-07-04,0.0,2023-07-04,Мегафон,Галина,Москва,0.0
182115,845,2023-07-04,0.0,2023-07-04,МТС,Галина,Воронеж,0.0
182338,850,2023-07-04,0.0,2023-07-04,МТС,Галина,Малый город,0.0
182568,851,2023-07-04,0.0,2023-07-04,МТС,Галина,Малый город,3.0
9821,290,2023-07-04,0.0,2023-07-04,Мегафон,Роман,Московская область,0.0
182831,856,2023-07-04,0.0,2023-07-04,МТС,Галина,Московская область,0.0
182889,862,2023-07-04,0.0,2023-07-04,Мегафон,Никита,Малый город,0.0
184013,863,2023-07-04,0.0,2023-07-04,Мегафон,Екатерина,Малый город,0.0
0,277,2023-07-04,0.0,2023-07-04,Мегафон,Екатерина,Санкт-Петербург,0.0


In [56]:
len(set(train_sparse['target_next_30d']))

67887

In [57]:
train_sparse = train_sparse[train_sparse['target_next_30d'].notna()]

In [58]:
def create_features(df):
    df = df.sort_values(['station_id', 'date']).copy()

    grouped = df.groupby('station_id')['daily_sales_count']

    # лаги (сколько продано N дней назад)
    for lag in [1, 7, 14, 28]:
        df[f'lag_{lag}'] = grouped.transform(lambda x: x.shift(lag))

    # скользящие статистики по прошлому (включая текущий день — он уже известен на момент отсчёта)
    for window in [7, 14, 30]:
        df[f'rolling_sum_{window}'] = grouped.transform(lambda x: x.rolling(window, min_periods=window).sum())
        df[f'rolling_std_{window}'] = grouped.transform(lambda x: x.rolling(window, min_periods=window).std())

    # возраст станции на момент отсчёта
    df['months_since_open'] = (df['date'] - df['created']).dt.days / 30.44

    # календарные признаки точки отсчёта
    df['day_of_week'] = df['date'].dt.dayofweek
    df['month'] = df['date'].dt.month

    return df

train_sparse = create_features(train_sparse)

In [59]:
train_sparse

,station_id,date,daily_sales_count,created,partner_name,employee_name,cities,target_next_30d,lag_1,lag_7,...,lag_28,rolling_sum_7,rolling_std_7,rolling_sum_14,rolling_std_14,rolling_sum_30,rolling_std_30,months_since_open,day_of_week,month
0,277,2023-07-04,0.0,2023-07-04,Мегафон,Екатерина,Санкт-Петербург,0.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,1,7
1,277,2023-07-05,0.0,2023-07-04,Мегафон,Екатерина,Санкт-Петербург,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.032852,2,7
2,277,2023-07-06,0.0,2023-07-04,Мегафон,Екатерина,Санкт-Петербург,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.065703,3,7
3,277,2023-07-07,0.0,2023-07-04,Мегафон,Екатерина,Санкт-Петербург,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.098555,4,7
4,277,2023-07-08,0.0,2023-07-04,Мегафон,Екатерина,Санкт-Петербург,1.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.131406,5,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1421967,3936,2026-06-30,0.0,2026-06-16,Inventive Group,Галина,Москва,0.0,0.0,0.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,0.459921,1,6
1421968,3936,2026-07-01,0.0,2026-06-16,Inventive Group,Галина,Москва,0.0,0.0,0.0,...,NaN,0.0,0.0,0.0,0.0,NaN,NaN,0.492773,2,7
1421999,3938,2026-06-29,0.0,2026-06-29,Inventive Group,Никита,Москва,12.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0,6
1422000,3938,2026-06-30,0.0,2026-06-29,Inventive Group,Никита,Москва,12.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.032852,1,6
